In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch

data_path = "../data/SN_d_tot_V2.0.csv"
column_names = ["year","month","day","decimal_date","sunspots","std_dev","n_observations","definitive"]
df = pd.read_csv(data_path, sep=";", header=None, names=column_names)

df_clean = df[df["sunspots"] != -1].reset_index(drop=True)
series = df_clean["sunspots"].values

In [3]:
split_idx = int(len(series) * 0.8)
train_data = series[:split_idx]
test_data = series[split_idx:]

In [4]:
print(len(train_data))
print(len(test_data))

58300
14575


In [7]:
# Min-Max-Normalisation

train_min = train_data.min()
train_max = train_data.max()

train_scaled = (train_data - train_min) / (train_max - train_min)

print(train_scaled.min())
print(train_scaled.max())

0.0
1.0


In [9]:
test_scaled = (test_data - train_min) / (train_max - train_min)

print(test_scaled.min())
print(test_scaled.max())

0.0
0.7765151515151515


In [11]:
def create_windows(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i : i + window_size])
        y.append(data[i + window_size])
    return np.array(X), np.array(y)

In [12]:
window_size = 30
X_train, y_train = create_windows(train_scaled, window_size)
X_test, y_test = create_windows(test_scaled, window_size)

print(X_train.shape, y_train.shape)

(58270, 30) (58270,)


In [18]:
X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1)
X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(-1)

y_train_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1,1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).reshape(-1,1)

In [19]:
print(X_train_t.shape, y_train_t.shape)
print(X_test_t.shape, y_test_t.shape)

torch.Size([58270, 30, 1]) torch.Size([58270, 1])
torch.Size([14545, 30, 1]) torch.Size([14545, 1])


In [20]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

for batch_X, batch_y in train_loader:
    print(batch_X.shape, batch_y.shape)
    break

torch.Size([64, 30, 1]) torch.Size([64, 1])


In [22]:
import torch.nn as nn

class SunspotLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, 1 )

    def forward(self,x):
        out, _ = self.lstm(x)
        out = out[:, -1 , :]
        out = self.fc(out)
        return out

In [23]:
model = SunspotLSTM()

for batch_X, batch_y in train_loader:
    test_output = model(batch_X)
    print(test_output.shape)
    break

torch.Size([64, 1])
